# Draw the run-389 hand mask

**Kernel:** use **Python 3 (ipykernel)** (Anaconda — has `ipympl`). *Not* the ana/psana env; this tool is psana-free.

The next cell opens an interactive figure: grey = run-389 sum image, **blue** = the automatic zero-mask floor (not editable), **red** = the hand regions you draw.

**Controls** (make sure the toolbar's pan/zoom button is *off* so clicks draw):
- `r` rectangle mode → click two opposite corners (auto-commits on the 2nd click)
- `p` polygon mode → click each vertex, then `enter` to close & commit
- `escape` cancel shape in progress · `u` undo last · `c` clear all
- `s` save · `q` save & quit

Saving writes `data/masks/human_Mask_run0389_{source,asm}.npy` (the target `methods.py` reads) + `hand_run0389.json` (your shapes — editable, and this notebook resumes from it).

In [ ]:
%matplotlib widget
%run producers/draw_hand_mask.py --run 389

Controls (click + keys; make sure the toolbar's pan/zoom is OFF so clicks draw):
  r : rectangle mode -- click two opposite corners (auto-commits on the 2nd click)
  p : polygon mode   -- click each vertex, then 'enter' to close & commit the polygon
  escape : cancel the shape in progress            u : undo last committed shape
  c : clear all hand shapes                         s : save    q : save & quit




In [ ]:
# After you've saved (s/q above): inspect the target you built (static, no widget needed)
import numpy as np, matplotlib.pyplot as plt
run = 389
sm   = np.load(f"data/images/sum_calib_run{run:04d}_asm.npy").astype(float)
tgt  = np.load(f"data/masks/human_Mask_run{run:04d}_asm.npy").astype(bool)
zero = np.load(f"data/masks/zero_mask_run{run:04d}_asm.npy").astype(bool)
real = sm[sm != 0]; disp = np.log10(np.clip(sm, np.percentile(real,30), np.percentile(real,99.7)))
ov = np.zeros((*tgt.shape, 4)); ov[zero] = (0.1,0.4,1,0.35); ov[tgt & ~zero] = (1,0,0,0.5)
fig, ax = plt.subplots(figsize=(9,9)); ax.imshow(disp, cmap="gray"); ax.imshow(ov)
ax.set_title(f"run {run}: target {100*tgt.mean():.2f}%  (blue=zero {100*zero.mean():.2f}%, "
             f"red=hand {100*(tgt&~zero).mean():.2f}%)"); ax.axis("off"); plt.show()
# Then score the automasker against it from a terminal:  python methods.py